# 04 - Data Quality

**Purpose:** This notebook checks whether the pipeline output is safe to use. It validates the Bronze, Silver, and Gold tables so that data issues are caught before the results are used.

**NOTE:** The notebook does not create or update amy table. It only runs checks and produces a pass or fail summary.

If any check fails, the notebook raises an error so the Databricks Job fails clearly.

**Source Tables:** 
- `workspace.bikepoint.bronze_bikepoint_raw`
- `workspace.bikepoint.silver_bikepoint`
- `workspace.bikepoint.gold_bikepoint_station_current_status`
- `workspace.bikepoint.gold_bikepoint_station_aggregated_metrics`

**Strategy:**

This notebook does not stop at the first failure. Every check runs unconditionally, even if a previous check fails. Results are collected into a list. At the end, it shows one complete summary of all checks, a summary table that prints every check with its pass/fail status. If any check failed, the notebook raises an exception.

**Why this approach?** 

We want a complete picture of pipeline health, not just the first thing that broke it. 

**Position in the pipeline**

This notebook runs as the **final task** in the Databricks Job DAG i.e. after Bronze -> Silver -> Gold have all completed. This notebook should run at the end of the Databricks Job, after all pipeline layers have been built. The pipeline order is:


```text
Bronze ingestion
        ↓
Silver transformation
        ↓
Gold table creation
        ↓
Data quality checks
```


In [0]:
CATALOG = "workspace"
SCHEMA = "bikepoint"

BRONZE_TABLE_FQN = f"{CATALOG}.{SCHEMA}.bronze_bikepoint_raw"
SILVER_TABLE_FQN = f"{CATALOG}.{SCHEMA}.silver_bikepoint"
GOLD_STATUS_FQN  = f"{CATALOG}.{SCHEMA}.gold_bikepoint_station_current_status"
GOLD_METRICS_FQN = f"{CATALOG}.{SCHEMA}.gold_bikepoint_station_aggregated_metrics"

# Store the result of every data quality check.
# Each check will add one dictionary to this list, which we would use to convert it into a summary report.
dq_results = []


def data_check(check_name, query_result, expected_value=0):
    status = "PASS" if query_result == expected_value else "FAIL"
    dq_results.append({
        "check": check_name,
        "expected": expected_value,
        "actual": query_result,
        "status": status,
    })
    print(f"{status}: {check_name} \nExpected: {expected_value}, Actual: {query_result}")
    print('---------------------------------------------------')

## Bronze checks

Bronze stores raw API responses as JSON strings. The checks here verify that ingestion produced sane records:
- No null `bikepoint_id` (Key)
- No null `raw_payload` (Actual Data)
- Every row has HTTP status 200
- `ingestion_ts` is in the past

In [0]:
# 'bikepoint_id' is the Key, and can not be NULL.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS null_rows
    FROM {BRONZE_TABLE_FQN}
    WHERE bikepoint_id IS NULL
""").collect()[0]["null_rows"]

data_check(
    "Bronze Layer: No NULL bikepoint_id",
    query_result,
)

# 'raw_payload' contains the actual data in JSON, and can never be NULL
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {BRONZE_TABLE_FQN}
    WHERE raw_payload IS NULL OR raw_payload = ''
""").collect()[0]["bad_records"]

data_check(
    "Bronze LAyer: No NULL or Empty raw_payload",
    query_result,
)

# HTTP status must be 200 because our Bronze notebook expects a successful GET response
result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {BRONZE_TABLE_FQN}
    WHERE http_status != 200
""").collect()[0]["bad_records"]

data_check(
    "Bronze: Rows have HTTP 200",
    result,
)

# ingestion_ts should never be in the future, especially if someone manually injecting test data
result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {BRONZE_TABLE_FQN}
    WHERE ingestion_ts > current_timestamp()
""").collect()[0]["bad_records"]

data_check(
    "Bronze: No Future Dated ingestion_ts",
    result,
)

## Silver checks

Silver is the cleaned and typed layer of the pipeline. At this stage, the raw JSON has already been parsed and important fields such as bike counts, dock counts, location, and station flags have been converted into proper columns.

These checks make sure the Silver table contains sensible station level data.

- bike, dock, and empty-dock counts are not negative
- available bikes plus empty docks do not exceed total docks
- No duplicates on the natural composite key `(bikepoint_id, pipeline_run_id)`
- occupancy rate is between `0` and `1` when it is available
- e-bike share is between `0` and `1` when it is available

In [0]:
# Bike and dock counts should never be negative, as that would not make business sense and may indicate bad parsing or bad source data.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {SILVER_TABLE_FQN}
    WHERE nb_bikes < 0
       OR nb_empty_docks < 0
       OR nb_docks < 0
       OR nb_standard_bikes < 0
       OR nb_ebikes < 0
""").collect()[0]["bad_records"]

data_check(
    "Silver Layer: No Negative bike or dock counts",
    query_result,
)

# The number of bikes plus empty docks should not be greater than the total number of docks.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {SILVER_TABLE_FQN}
    WHERE is_consistent = false
       OR is_consistent IS NULL
""").collect()[0]["bad_records"]

data_check(
    "Silver Layer: Capacity Values are consistent",
    query_result,
)


# Each station should not have more than one row for each pipeline run
query_result = spark.sql(f"""
    SELECT COUNT(*) AS duplicate_records
    FROM (
        SELECT
            bikepoint_id,
            pipeline_run_id,
            COUNT(*) AS row_count
        FROM {SILVER_TABLE_FQN}
        GROUP BY bikepoint_id, pipeline_run_id
        HAVING COUNT(*) > 1
    )
""").collect()[0]["duplicate_records"]

data_check(
    "Silver Layer: No Duplicates Found",
    query_result,
)


# occupancy_rate should be between 0 and 1, with 0 means empty, 1 means fully occupied with bikes.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {SILVER_TABLE_FQN}
    WHERE occupancy_rate IS NOT NULL
      AND (occupancy_rate < 0 OR occupancy_rate > 1)
""").collect()[0]["bad_records"]

data_check(
    "Silver Layer: Occupancy Rate is between 0 and 1",
    query_result,
)

# ebike_share should be between 0 and 1, with 0.25 means 25% of available bikes are e-bikes.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {SILVER_TABLE_FQN}
    WHERE ebike_share IS NOT NULL
      AND (ebike_share < 0 OR ebike_share > 1)
""").collect()[0]["bad_records"]

data_check(
    "Silver Layer: E-bike share is between 0 and 1",
    query_result,
)

## Gold checks

Gold is the business-ready layer of the pipeline. At this stage, the data has already been cleaned, aggregated, and labelled so it can be used for reporting, dashboards, and decision making.

These checks make sure the Gold tables are reliable and easy to trust. We verify:

- Both Gold tables have exactly one row per station with no duplicates
- `availability_risk_score` is in [0, 1]
- `availability_risk_band` only contains the allowed values (`critical`, `high`, `medium`, `low`)
- `capacity_category` only contains the allowed values (`small`, `medium`, `large`)

In [0]:
# The Gold current_status table should have only one row per station, otherwise the latest snapshot logic may not be working correctly.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS duplicate_records
    FROM (
        SELECT
            bikepoint_id,
            COUNT(*) AS row_count
        FROM {GOLD_STATUS_FQN}
        GROUP BY bikepoint_id
        HAVING COUNT(*) > 1
    )
""").collect()[0]["duplicate_records"]

data_check(
    "Gold Status Layer: No Duplicates",
    query_result,
)


# The Gold aggregated_metrics table should also have only one row per station.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS duplicate_records
    FROM (
        SELECT
            bikepoint_id,
            COUNT(*) AS row_count
        FROM {GOLD_METRICS_FQN}
        GROUP BY bikepoint_id
        HAVING COUNT(*) > 1
    )
""").collect()[0]["duplicate_records"]

data_check(
    "Gold Metrics Layer:  No Duplicates",
    query_result,
)


# availability_risk_score should always be between 0 and 1.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {GOLD_METRICS_FQN}
    WHERE availability_risk_score IS NULL
       OR availability_risk_score < 0
       OR availability_risk_score > 1
""").collect()[0]["bad_records"]

data_check(
    "Gold Metrics Layer: Availability Risk Score is between 0 and 1",
    query_result,
)

# reliability_score should equal 1 - availability_risk_score
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {GOLD_METRICS_FQN}
    WHERE ABS(reliability_score - (1.0 - availability_risk_score)) > 0.02
""").collect()[0]["bad_records"]

data_check(
    "Gold Metrics Layer: Reliability Score equals 1 - availability_risk_score",
    query_result,
)

# availability_risk_band should only contain the labels we created in the Gold logic.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {GOLD_METRICS_FQN}
    WHERE availability_risk_band IS NULL
       OR availability_risk_band NOT IN ('critical', 'high', 'medium', 'low')
""").collect()[0]["bad_records"]

data_check(
    "Gold Metrics Layer: Availability Risk Score is between 0 and 1",
    query_result,
)


# capacity_category should only contain small, medium, or large groups.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS bad_records
    FROM {GOLD_METRICS_FQN}
    WHERE capacity_category IS NULL
       OR capacity_category NOT IN ('small', 'medium', 'large')
""").collect()[0]["bad_records"]

data_check(
    "Gold Metrics Layer: Capacity Category has valid values",
    query_result,
)

## Cross Layer Checks

Cross layer check compares the pipeline layers with each other. The goal is to make sure that data has moved correctly from Bronze to Silver to Gold. These checks help confirm that no stations were accidentally lost, duplicated, or created incorrectly during transformation.

We validate that:

- the number of distinct stations in Silver matches Bronze
- every station in the Gold tables also exists in Silver
- both Gold tables contain a consistent number of stations

These checks are important because each layer depends on the previous one.

In [0]:
# Bronze and Silver should contain the same distinct stations, that checks whether any station was lost during the Bronze-to-Silver transformation.
query_result = spark.sql(f"""
    WITH bronze_stations AS (
        SELECT DISTINCT bikepoint_id
        FROM {BRONZE_TABLE_FQN}
    ),

    silver_stations AS (
        SELECT DISTINCT bikepoint_id
        FROM {SILVER_TABLE_FQN}
    )

    SELECT
        ABS(
            (SELECT COUNT(*) FROM bronze_stations)
            -
            (SELECT COUNT(*) FROM silver_stations)
        ) AS station_count_difference
""").collect()[0]["station_count_difference"]

data_check(
    "Cross Layer: Bronze and Silver station counts match",
    query_result,
)


# Every station in the Gold current_status table should exist in Silver.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS orphan_records
    FROM {GOLD_STATUS_FQN} gold
    LEFT ANTI JOIN {SILVER_TABLE_FQN} silver
        ON gold.bikepoint_id = silver.bikepoint_id
""").collect()[0]["orphan_records"]

data_check(
    "Cross Layer: No Orphan Stations in the Gold Corrent Layer",
    query_result,
)


# Every station in the Gold aggregated_metrics table should also exist in Silver.
query_result = spark.sql(f"""
    SELECT COUNT(*) AS orphan_records
    FROM {GOLD_METRICS_FQN} gold
    LEFT ANTI JOIN {SILVER_TABLE_FQN} silver
        ON gold.bikepoint_id = silver.bikepoint_id
""").collect()[0]["orphan_records"]

data_check(
    "Cross Layer: No Orphan Stations in Gold Aggregated Metrics",
    query_result,
)


# Both Gold tables should have the same number of rows.
status_count = spark.sql(f"""
    SELECT COUNT(*) AS row_count
    FROM {GOLD_STATUS_FQN}
""").collect()[0]["row_count"]

metrics_count = spark.sql(f"""
    SELECT COUNT(*) AS row_count
    FROM {GOLD_METRICS_FQN}
""").collect()[0]["row_count"]

query_result = abs(status_count - metrics_count)

data_check(
    "Cross Layer: Gold Current Status and Gold Aggregated Metrics row counts match",
    query_result,
)

In [0]:

# This section displays the Final Data Quality Report. If any check failed, the notebook raises an error so the Databricks Job fails clearly, and we would get an alert via email.
import pandas as pd

# Convert the list of results into a DataFrame
summary_df = pd.DataFrame(dq_results)
summary_df = summary_df[["status", "check", "expected", "actual"]]

# Count Total, Passed, and Failed checks.
total_checks = len(summary_df)
passed_checks = (summary_df["status"] == "PASS").sum()
failed_checks = (summary_df["status"] == "FAIL").sum()

print("-" * 60)
print("DATA QUALITY REPORT")
print("-" * 60)
print(f"Total Checks: {total_checks}")
print(f"Passed Checks: {passed_checks}")
print(f"Failed Checks: {failed_checks}")
print("-" * 60)
print()

In [0]:
display(summary_df.head())

In [0]:
# Fail the notebook if any check FAILED, instead of silently accepting any bad data
if failed_checks > 0:
    failed_check_names = summary_df.loc[
        summary_df["status"] == "FAIL",
        "check"
    ].tolist()

    raise Exception(
        f"DATA QUALITY FAILED: {failed_checks} out of {total_checks} checks FAILED!!\n"
        f"Failed checks:\n- " + "\n- ".join(failed_check_names)
    )
print("All Data Quality Checks PASSED!!")